In [44]:
# OPUS‑MT FINE‑TUNING + MINERVA REFINEMENT
# --------------------------------------------------
!pip install --no-deps xformers trl peft accelerate bitsandbytes sacremoses -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 12.6 MB/s eta 0:00:0000:010:01


In [2]:
import pandas as pd
import torch
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import requests
from lxml import html
from bs4 import BeautifulSoup


2025-06-19 17:53:20.365394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750355600.563640      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750355600.620288      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [34]:
def extract_paragraphs(raw_html):
    """Extracts and cleans paragraph text from translation-entry HTML."""
    soup = BeautifulSoup(raw_html, 'html.parser')
    paragraphs = soup.find_all('p')
    results = []
    for p in paragraphs:
        # Remove line numbers (span.ln)
        for span in p.find_all('span', class_='ln'):
            span.decompose()
        # Flatten to single line and clean
        text = p.get_text(separator=' ', strip=True).replace('\n', ' ')
        results.append(text)
    return results

def extract_links_from_xpath(tree, xpath):
    """Extract all hrefs from <a> tags inside <ul> from a given XPath."""
    nodes = tree.xpath(xpath)
    links = []
    for ul in nodes:
        hrefs = ul.xpath('.//li/a/@href')
        links.extend(hrefs)
    return links

def scrape_divine_comedy_links():
    url = "https://digitaldante.columbia.edu/dante/divine-comedy/"
    response = requests.get(url)
    tree = html.fromstring(response.content)

    # XPaths for Inferno, Purgatorio, Paradiso
    xpaths = [
        "/html/body/main/div[1]/ul",
        "/html/body/main/div[2]/ul",
        "/html/body/main/div[3]/ul",
    ]

    all_links = []
    for xpath in xpaths:
        section_links = extract_links_from_xpath(tree, xpath)
        all_links.extend(section_links)

    return all_links

def scrape_and_pair_verses(url):
    response = requests.get(url)
    tree = html.fromstring(response.content)

    # Define XPaths for Italian and English
    xpath_italian = "/html/body/main/div[3]/div/section/div[2]/div/div/div[1]/section/div[1]/div"
    xpath_english = "/html/body/main/div[3]/div/section/div[2]/div/div/div[2]/section/div[1]/div"

    node_italian = tree.xpath(xpath_italian)
    node_english = tree.xpath(xpath_english)

    if not node_italian or not node_english:
        print(f"Warning: content not found for {url}")
        return []

    html_italian = html.tostring(node_italian[0], encoding='unicode', method='html')
    html_english = html.tostring(node_english[0], encoding='unicode', method='html')

    italian_paragraphs = extract_paragraphs(html_italian)
    english_paragraphs = extract_paragraphs(html_english)

    if len(italian_paragraphs) != len(english_paragraphs):
        print(f"Mismatch in paragraph count for {url} (IT: {len(italian_paragraphs)}, EN: {len(english_paragraphs)})")

    return list(zip(italian_paragraphs, english_paragraphs))

def read_dante_csv():
    with open("/kaggle/working/divine_comedy_verses.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)  # <-- key fix
        writer.writerow(["link", "italian", "english"])
        writer.writerows(all_pairs)

        df = pd.read_csv(
            "divine_comedy_verses.csv",
            quotechar='"',
            escapechar='\\',
            on_bad_lines='skip'  # Optional: to skip any remaining malformed lines
        )[["english", "italian"]]
        return df

if not os.path.exists("/kaggle/input/divine-comedy-english/divine_comedy_verses(2).csv"):
    links = scrape_divine_comedy_links()
    print(f"Found {len(links)} canto links.")

    all_pairs = []
    for link in links:
        print(f"Processing: {link}")
        pairs = scrape_and_pair_verses(link)
        all_pairs.extend([(link, it, en) for it, en in pairs])

    with open("divine_comedy_verses.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["link", "italian", "english"])
    
        for link, it, en in all_pairs:
            # Basic validation
            if not it or not en:
                continue  # skip if one is missing
            try:
                it_clean = it.replace('\n', ' ').replace('\r', ' ').strip()
                en_clean = en.replace('\n', ' ').replace('\r', ' ').strip()
                writer.writerow([link.strip(), it_clean, en_clean])
            except Exception as e:
                print(f"Skipping malformed row: {e}")
    print("Saved to divine_comedy_verses.csv")

df = read_dante_csv()

In [51]:
# --- Dataset ---
raw_ds = Dataset.from_pandas(df)
split = raw_ds.train_test_split(test_size=0.1)
train_ds, val_ds = split["train"], split["test"]

model_it_en_ckpt = "Helsinki-NLP/opus-mt-it-en"
tokenizer_it_en = AutoTokenizer.from_pretrained(model_it_en_ckpt)

def preprocess(batch):
    model_inputs = tokenizer_it_en(
        batch["italian"],
        padding="max_length",
        truncation=True,
        max_length=128,
        text_target=batch["english"]
    )
    return model_inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_ds  = val_ds.map(preprocess,  batched=True, remove_columns=val_ds.column_names)

model_it_en = AutoModelForSeq2SeqLM.from_pretrained(model_it_en_ckpt, device_map="auto")

args = Seq2SeqTrainingArguments(
    output_dir="./results_opus",
    per_device_train_batch_size=32,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_steps=10,
    save_total_limit=3,
    predict_with_generate=True,
    eval_strategy="steps",   
    eval_steps=20,    
    load_best_model_at_end=True,             
    metric_for_best_model="eval_loss",       
    greater_is_better=False,                 
    report_to="none",
)


trainer = Seq2SeqTrainer(model=model_it_en, args=args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tokenizer_it_en)
print("▶ Fine‑tuning OPUS‑MT ...")
trainer.train()
print("✅ Done!")

Map:   0%|          | 0/4314 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

/tmp/ipykernel_35/751191071.py:41: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(model=model_it_en, args=args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tokenizer_it_en)


▶ Fine‑tuning OPUS‑MT ...


Step,Training Loss,Validation Loss
20,1.015900,0.877373
40,0.863400,0.821448
60,0.833700,0.799403
80,0.823300,0.783862
100,0.816700,0.771064
120,0.803700,0.761801
140,0.761200,0.750935
160,0.753600,0.742439
180,0.728300,0.733454
200,0.738200,0.725722


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[80378]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


✅ Done!


In [61]:
model.save_pretrained("./model_it_en_local")
tokenizer.save_pretrained("./model_it_en_local")

('./model_it_en_local/tokenizer_config.json',
 './model_it_en_local/special_tokens_map.json',
 './model_it_en_local/vocab.json',
 './model_it_en_local/source.spm',
 './model_it_en_local/target.spm',
 './model_it_en_local/added_tokens.json')

In [59]:
model_en_it_ckpt = "Helsinki-NLP/opus-mt-en-it"
tokenizer_en_it = AutoTokenizer.from_pretrained(model_en_it_ckpt)
model_en_it = AutoModelForSeq2SeqLM.from_pretrained(model_en_it_ckpt, device_map="auto")

class Evaluate:
    def __init__(self, model_en_it, tokenizer_en_it, model_it_en, tokenizer_it_en):
        self.model_en_it = model_en_it
        self.tokenizer_en_it = tokenizer_en_it
        self.model_it_en = model_it_en
        self.tokenizer_it_en = tokenizer_it_en

    def translate(self, text, tokenizer, model):
        model.eval()
        with torch.no_grad():
            # Tokenize input
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
            # Move tensors to the same device as the model
            model_device = next(model.parameters()).device
            inputs = {k: v.to(model_device) for k, v in inputs.items()}
            # Generate translation
            outputs = model.generate(**inputs)
            return tokenizer.decode(outputs[0], skip_special_tokens=True)

    def round_trip(self, italian_phrase):
        # Italian → English
        english_translation = self.translate(
            italian_phrase,
            self.tokenizer_it_en,
            self.model_it_en
        )
        # English → Italian
        italian_back = self.translate(
            english_translation,
            self.tokenizer_en_it,
            self.model_en_it
        )
        return {
            "original_italian": italian_phrase,
            "to_english": english_translation,
            "back_to_italian": italian_back
        }

evaluator = Evaluate(model_en_it, tokenizer_en_it, model_it_en, tokenizer_it_en)

In [60]:
# --- Inference ---
sample = "sia in mezzo tra me e te: con noi non puo' tu già più lungamente dimorare, ch'io non lo sofferrò e non lo lascerò."
evaluator.round_trip(sample)

{'original_italian': "sia in mezzo tra me e te: con noi non puo' tu già più lungamente dimorare, ch'io non lo sofferrò e non lo lascerò.",
 'to_english': 'may it be between you and me; you cannot dwell with us any longer than I have endured, and I shall not leave him.',
 'back_to_italian': 'sia tra voi e me; voi non potete abitare con noi più a lungo di quanto io abbia sopportato, e io non lo lascerò.'}

In [72]:
test_df = pd.read_excel("/kaggle/input/dataset/dataset.xlsx")
predictions = []
for sample in test_df["Sentence"]:
    t = evaluator.round_trip(sample)
    print(t)
    predictions.append(t["back_to_italian"])
test_df["prediction"] = predictions

{'original_italian': "quella guerra ben fatta l' opera perché etc. Et dall' altra parte Aiaces era uno cavaliere franco e prode all' arme, di gran guisa, ma non era pieno di grande senno", 'to_english': 'that war done well because, etc. Et, on the other side, Aiaces was a knight who was free and valiant to arms, with great force, but was not full of great sense', 'back_to_italian': "che la guerra ha fatto bene perché, ecc. Et, dall'altra parte, Aiaces era un cavaliere che era libero e valoroso alle armi, con grande forza, ma non era pieno di grande senso"}
{'original_italian': 'crudele, e di tutte le colpe pigli vendetta, come dice la legge, ed a neuno cavaliere perdoni che pecchi.', 'to_english': 'cruel, and revenge for all sins, as the law says, and forgive no knight for sinning.', 'back_to_italian': 'crudele, e vendetta per tutti i peccati, come dice la legge, e non perdonare nessun cavaliere per il peccato.'}
{'original_italian': "Non d' altra forza d' animo fue ornato Ponzio Aufid

In [74]:
test_df["prediction"] = predictions

In [78]:
test_df.to_csv("predicted_opus.csv")